# 🎬 OmniVoice AI: Dublador & Clonador de Voz Multilíngue com Sincronização de Tempo
### Envie seu vídeo MP4 ou áudio, clone sua própria voz e exporte o conteúdo dublado exatamente no mesmo tempo do original!

Este notebook executa um pipeline completo e profissional de localização de vídeo e áudio:
1. **Entrada Universal:** Suporte para arquivos de vídeo (`.mp4`, `.mov`, `.mkv`) ou áudio (`.wav`, `.mp3`, `.m4a`) e microfone.
2. **Reconhecimento de Fala (Whisper):** Transcreve com alta precisão e detecta automaticamente o idioma original.
3. **Tradução Automática Segmentada:** Tradução robusta sem limite de caracteres (**Inglês, Espanhol, Francês, Alemão, Chinês, Árabe** e mais).
4. **Clonagem de Voz com IA (OmniVoice):** Recria a sua voz falando no idioma de destino mantendo timbre, entonação e características vocais únicas.
5. **Sincronização Temporal com o Vídeo Original:** Aplica time-stretching inteligente com preservação total de tom (*pitch-preserved atempo* via FFmpeg), garantindo que o áudio dublado case perfeitamente com a duração do vídeo.
6. **Exportação Dupla:** Gera o **áudio sincronizado** e o **vídeo MP4 final** com a trilha de áudio substituída sem perda de qualidade visual.

> ⚠️ **Requisito Obrigatório (GPU T4):**
> Vá no menu superior do Colab em **Ambiente de Execução (Runtime)** ➔ **Alterar tipo de ambiente de execução (Change runtime type)** ➔ Selecione **T4 GPU**.

In [ ]:
# @title Passo 1: Instalar Dependências e FFmpeg
# @markdown Instala OmniVoice, Whisper, Gradio, Deep-Translator e ferramentas de mídia.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg
!pip install -q omnivoice gradio openai-whisper deep-translator
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
print('✅ Dependências e ferramentas de mídia instaladas com sucesso!')

In [ ]:
# @title Passo 2: Carregar os Modelos de IA na GPU (T4 / A100 / L4)
# @markdown Baixa e carrega o Whisper e o OmniVoice na memória de vídeo da GPU.

import os
import torch
import torchaudio
import whisper
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'🖥️ Dispositivo em uso: {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU Detectada: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ GPU não detectada! Por favor, ative a T4 GPU no menu do Colab (Runtime > Change runtime type).')

print('\n⏳ Carregando modelo Whisper (base) para transcrição de áudio...')
whisper_model = whisper.load_model('base', device=device)
print('✅ Whisper pronto!')

print('\n⏳ Carregando OmniVoice da k2-fsa (primeira vez baixa os pesos do modelo)...')
omnivoice_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ OmniVoice carregado e pronto para clonar sua voz!')

In [ ]:
# @title Passo 3: Motor de Processamento, Sincronização Temporal e Remuxing de Vídeo
# @markdown Funções auxiliares para extração de áudio, medição de duração, ajuste atempo e geração do MP4 final.

import subprocess
import json
import tempfile
import os
import re
import torch
import torchaudio
from deep_translator import GoogleTranslator

def get_media_info(file_path):
    """Retorna a duração em segundos e se o arquivo contém faixa de vídeo."""
    if not file_path or not os.path.exists(file_path):
        return 0.0, False
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration:stream=codec_type',
        '-of', 'json', file_path
    ]
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(res.stdout)
        duration = float(data.get('format', {}).get('duration', 0.0))
        streams = data.get('streams', [])
        has_video = any(s.get('codec_type') == 'video' for s in streams)
        return duration, has_video
    except Exception as e:
        print(f'Erro ao inspecionar mídia com ffprobe: {e}')
        return 0.0, False

def extract_audio_to_wav(media_path, output_wav):
    """Converte qualquer mídia (vídeo ou áudio) em WAV 24kHz mono para o OmniVoice."""
    cmd = [
        'ffmpeg', '-y', '-i', media_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def extract_audio_slice(input_wav, start_sec, end_sec, output_slice_wav):
    """Extrai uma fatia de áudio (ex: 5 a 12s) para servir de referência ideal ao OmniVoice."""
    duration = max(0.5, end_sec - start_sec)
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_sec:.3f}',
        '-t', f'{duration:.3f}',
        '-i', input_wav,
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_slice_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def translate_text_robust(text, target_code, max_chunk=2500):
    """
    Traduz textos de qualquer tamanho sem estourar o limite de 5000 caracteres do GoogleTranslator.
    Divide inteligentemente por pontuação e parágrafos.
    """
    if not text or not text.strip():
        return ''
    clean_text = text.strip()
    if len(clean_text) < max_chunk:
        return GoogleTranslator(source='auto', target=target_code).translate(clean_text)

    # Divide por pontuação natural de fim de frase
    sentences = re.split(r'(?<=[.!?\n])\s+', clean_text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    translator = GoogleTranslator(source='auto', target=target_code)
    translated_parts = []
    for chunk in chunks:
        chunk = chunk.strip()
        if chunk:
            part = translator.translate(chunk)
            if part:
                translated_parts.append(part)
    return ' '.join(translated_parts)

def build_atempo_filter(speed_factor):
    """Gera encadeamento de filtros atempo no FFmpeg (cada filtro suporta entre 0.5 e 2.0)."""
    speed = speed_factor
    filters = []
    while speed > 2.0:
        filters.append('atempo=2.0')
        speed /= 2.0
    while speed < 0.5:
        filters.append('atempo=0.5')
        speed /= 0.5
    filters.append(f'atempo={speed:.5f}')
    return ','.join(filters)

def time_sync_audio(synth_wav_path, target_duration, output_synced_wav):
    """
    Ajusta a velocidade do áudio gerado para casar exatamente com a duração do original,
    preservando o tom e timbre da voz clonada (pitch-preserved time stretch).
    """
    synth_duration, _ = get_media_info(synth_wav_path)
    if synth_duration <= 0 or target_duration <= 0:
        cmd = ['ffmpeg', '-y', '-i', synth_wav_path, '-c', 'copy', output_synced_wav]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return 1.0, synth_duration

    speed_factor = synth_duration / target_duration

    # Se a diferença de duração for mínima (< 1.5%), mantém ritmo natural e ajusta preenchimento
    if 0.985 <= speed_factor <= 1.015:
        filter_chain = f'apad=whole_dur={target_duration:.4f}'
    else:
        tempo_filter = build_atempo_filter(speed_factor)
        filter_chain = f'{tempo_filter},apad=whole_dur={target_duration:.4f}'

    cmd = [
        'ffmpeg', '-y', '-i', synth_wav_path,
        '-filter:a', filter_chain,
        '-t', f'{target_duration:.4f}',
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_synced_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return speed_factor, synth_duration

def remux_video_with_audio(original_video_path, new_audio_path, output_video_path):
    """
    Substitui a faixa de áudio do vídeo original pelo áudio dublado e sincronizado.
    Utiliza stream copy (-c:v copy) para renderização instantânea sem perda de qualidade visual.
    """
    cmd = [
        'ffmpeg', '-y',
        '-i', original_video_path,
        '-i', new_audio_path,
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-b:a', '192k',
        '-shortest',
        output_video_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def generate_voice_cloning_chunked(model, text, ref_audio, ref_text, num_steps=32, speed=1.0, max_chunk_chars=300):
    """
    Gera fala com OmniVoice dividindo textos longos em frases naturais para máxima fidelidade,
    evitando estouro de VRAM na GPU e garantindo entonação constante.
    """
    if len(text) <= max_chunk_chars:
        out = model.generate(text=text, ref_audio=ref_audio, ref_text=ref_text, num_step=int(num_steps), speed=float(speed))
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        return t.unsqueeze(0) if t.dim() == 1 else t

    # Divide em frases naturais preservando pontuação
    sentences = re.split(r'(?<=[.!?;\\n])\\s+', text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk_chars:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    print(f'🎙️ Sintetizando áudio em {len(chunks)} blocos naturais de fala...')
    tensors = []
    silence = torch.zeros((1, int(24000 * 0.15)))
    for idx, c in enumerate(chunks):
        print(f'  - Gerando bloco {idx+1}/{len(chunks)}: {c[:45]}...')
        out = model.generate(text=c, ref_audio=ref_audio, ref_text=ref_text, num_step=int(num_steps), speed=float(speed))
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        if t.dim() == 1:
            t = t.unsqueeze(0)
        tensors.append(t)
        tensors.append(silence)

    if tensors:
        tensors.pop()
        return torch.cat(tensors, dim=-1)
    return torch.zeros((1, 24000))

print('✅ Motor de processamento, tradução robusta e sincronização carregado!')

In [ ]:
# @title Passo 4: Iniciar a Interface de Dublagem Sincronizada (Gradio)
# @markdown Clique no botão 'Play' e acesse o link público 'Running on public URL: https://...gradio.live'

import gradio as gr

# Idiomas suportados com destaque para os 6 principais solicitados
LANGUAGES = {
    '🇺🇸 Inglês (English)': 'en',
    '🇪🇸 Espanhol (Español)': 'es',
    '🇫🇷 Francês (Français)': 'fr',
    '🇩🇪 Alemão (Deutsch)': 'de',
    '🇨🇳 Chinês Simplificado (中文)': 'zh-CN',
    '🇸🇦 Árabe (العربية)': 'ar',
    '🇧🇷 Português (Português)': 'pt',
    '🇮🇹 Italiano (Italiano)': 'it',
    '🇯🇵 Japonês (日本語)': 'ja',
    '🇷🇺 Russo (Русский)': 'ru'
}

def process_dubbing(media_file, target_lang_label, sync_duration_opt, num_steps, user_speed):
    if not media_file:
        return (
            None, None,
            '❌ **Erro:** Por favor, envie um arquivo de vídeo (.mp4, .mov, etc.) ou de áudio (.wav, .mp3, etc.).',
            '', ''
        )

    try:
        target_code = LANGUAGES[target_lang_label]
        orig_duration, has_video = get_media_info(media_file)

        # 1. Extrair áudio completo para WAV a 24kHz
        temp_full_wav = tempfile.NamedTemporaryFile(suffix='_full.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_full_wav)

        # 2. Transcrever áudio original com Whisper
        print('🎙️ Transcrevendo áudio original com Whisper...')
        asr_result = whisper_model.transcribe(temp_full_wav)
        original_text = asr_result.get('text', '').strip()
        detected_lang = asr_result.get('language', 'desconhecido')
        segments = asr_result.get('segments', [])

        if not original_text:
            return (
                None, None,
                '❌ **Erro:** Não foi possível reconhecer fala audível no arquivo enviado.',
                '', ''
            )

        # 3. Selecionar amostra ideal de referência vocal para o OmniVoice (3 a 12 segundos)
        # O OmniVoice foi projetado para usar um clipe curto e limpo como prompt acústico
        ref_slice_wav = tempfile.NamedTemporaryFile(suffix='_ref_slice.wav', delete=False).name
        ref_slice_start = 0.0
        ref_slice_end = min(orig_duration, 10.0)
        ref_slice_text = original_text[:120]

        if segments:
            seg_texts = []
            ref_slice_start = segments[0]['start']
            ref_slice_end = segments[0]['end']
            for s in segments:
                seg_texts.append(s['text'].strip())
                ref_slice_end = s['end']
                if (ref_slice_end - ref_slice_start) >= 5.0:
                    break
            ref_slice_text = ' '.join(seg_texts)

        extract_audio_slice(temp_full_wav, ref_slice_start, ref_slice_end, ref_slice_wav)

        # 4. Traduzir o texto com proteção contra limites de tamanho (> 5000 caracteres)
        print(f'🌍 Traduzindo do {detected_lang} para {target_lang_label} (Texto total: {len(original_text)} caracteres)...')
        translated_text = translate_text_robust(original_text, target_code)

        # 5. Sintetizar fala clonando a voz original com OmniVoice em blocos naturais
        print('🧬 Sintetizando fala clonada com OmniVoice...')
        audio_tensor = generate_voice_cloning_chunked(
            model=omnivoice_model,
            text=translated_text,
            ref_audio=ref_slice_wav,
            ref_text=ref_slice_text,
            num_steps=int(num_steps),
            speed=float(user_speed)
        )

        # Salvar áudio cru sintetizado
        temp_synth_wav = tempfile.NamedTemporaryFile(suffix='_synth.wav', delete=False).name
        torchaudio.save(temp_synth_wav, audio_tensor.cpu(), 24000)

        # 6. Sincronização Temporal (Mesmo tempo do vídeo original)
        final_audio_path = tempfile.NamedTemporaryFile(suffix='_dubbed.wav', delete=False).name
        speed_factor = 1.0
        synth_duration = audio_tensor.shape[-1] / 24000.0

        if sync_duration_opt and orig_duration > 0:
            print(f'⏱️ Sincronizando duração: {synth_duration:.2f}s ➔ {orig_duration:.2f}s...')
            speed_factor, synth_duration = time_sync_audio(temp_synth_wav, orig_duration, final_audio_path)
            final_duration = orig_duration
        else:
            final_audio_path = temp_synth_wav
            final_duration = synth_duration

        # 7. Se a entrada for vídeo, gerar o vídeo MP4 dublado
        final_video_path = None
        if has_video:
            print('🎬 Integrando áudio dublado ao vídeo original (MP4)...')
            final_video_path = tempfile.NamedTemporaryFile(suffix='_dubbed.mp4', delete=False).name
            remux_video_with_audio(media_file, final_audio_path, final_video_path)

        status_md = f"""
### ✅ Dublagem Concluída com Sucesso!
- 🌐 **Idioma Original Detectado:** `{detected_lang.upper()}`
- 🎯 **Idioma de Destino:** `{target_lang_label}`
- ⏱️ **Duração do Original:** `{orig_duration:.2f}s`
- 🔊 **Duração Sintetizada (Bruta):** `{synth_duration:.2f}s`
- ⚡ **Duração Final Sincronizada:** `{final_duration:.2f}s`
- 🎛️ **Fator de Ajuste de Tempo (atempo):** `{speed_factor:.3f}x`
- 🎞️ **Formato de Saída:** `{'Vídeo MP4 + Áudio WAV' if has_video else 'Áudio WAV'}`
"""
        return final_audio_path, final_video_path, status_md, original_text, translated_text

    except Exception as e:
        import traceback
        traceback.print_exc()
        return (
            None, None,
            f'❌ **Erro durante a execução:** `{str(e)}`',
            '', ''
        )

def process_free_tts(custom_text, ref_audio, num_steps, speed):
    if not custom_text.strip():
        return None, '❌ Digite um texto para sintetizar.'
    if not ref_audio:
        return None, '❌ Envie uma amostra de áudio com a voz a ser clonada.'
    try:
        audio_tensor = generate_voice_cloning_chunked(
            model=omnivoice_model,
            text=custom_text,
            ref_audio=ref_audio,
            ref_text='',
            num_steps=int(num_steps),
            speed=float(speed)
        )
        tmp_file = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        torchaudio.save(tmp_file.name, audio_tensor.cpu(), 24000)
        return tmp_file.name, '✅ Fala sintetizada com sucesso com a sua voz!'
    except Exception as e:
        return None, f'❌ Erro: {str(e)}'

# Interface Gráfica Moderna
with gr.Blocks(title='OmniVoice AI Voice & Video Dubber', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue')) as demo:
    gr.HTML("""
    <div style='text-align: center; margin-bottom: 18px;'>
        <h1 style='font-size: 2.2em; margin-bottom: 6px;'>🎙️ OmniVoice — Dublagem Sincronizada de Vídeo & Áudio</h1>
        <p style='color: #555; font-size: 1.1em;'>
            Clone sua voz e duble qualquer vídeo MP4 ou áudio para <b>Inglês, Espanhol, Francês, Alemão, Chinês e Árabe</b>
            mantendo <b>exatamente o mesmo tempo de duração</b> do vídeo original!
        </p>
    </div>
    """)

    with gr.Tabs():
        # --- ABA 1: DUBLAGEM SINCRONIZADA ---
        with gr.TabItem('🎬 Dublagem Sincronizada (Vídeo ou Áudio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_media = gr.File(
                        label='1. Envie seu Vídeo (MP4, MOV, MKV) ou Áudio (WAV, MP3, M4A)',
                        file_count='single',
                        type='filepath'
                    )
                    target_lang = gr.Dropdown(
                        choices=list(LANGUAGES.keys()),
                        value='🇺🇸 Inglês (English)',
                        label='2. Idioma de Destino da Dublagem'
                    )
                    sync_checkbox = gr.Checkbox(
                        value=True,
                        label='⏱️ Sincronizar Duração com o Original (Garante mesmo tempo do vídeo)',
                        info='Acelera ou desacelera suavemente a fala mantendo o tom natural da voz clonada.'
                    )
                    with gr.Accordion('⚙️ Configurações Avançadas de IA', open=False):
                        steps_slider = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão (Diffusion Steps)')
                        speed_slider = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label='Velocidade Base da Fala')
                    btn_dub = gr.Button('✨ Dublar e Sincronizar Vídeo/Áudio', variant='primary', size='lg')
                    status_label = gr.Markdown('')

                with gr.Column(scale=1):
                    output_audio = gr.Audio(label='🔊 Áudio Dublado Sincronizado (Com a sua voz)', type='filepath', interactive=False)
                    output_video = gr.Video(label='🎬 Vídeo Dublado Final (MP4 Sincronizado)', interactive=False)
                    with gr.Accordion('📜 Transcrição e Tradução', open=True):
                        txt_orig = gr.Textbox(label='Transcrição Original (Whisper)', lines=3, interactive=False)
                        txt_trans = gr.Textbox(label='Tradução Gerada para Dublagem', lines=3, interactive=False)

            btn_dub.click(
                fn=process_dubbing,
                inputs=[input_media, target_lang, sync_checkbox, steps_slider, speed_slider],
                outputs=[output_audio, output_video, status_label, txt_orig, txt_trans]
            )

        # --- ABA 2: CLONAGEM LIVRE ---
        with gr.TabItem('✍️ Clonagem Livre (Digitar Texto Personalizado)'):
            gr.Markdown('Envie uma amostra de áudio com a sua voz e digite qualquer texto em qualquer idioma para sintetizar diretamente:')
            with gr.Row():
                with gr.Column(scale=1):
                    ref_audio_free = gr.Audio(label='Áudio de Referência (sua voz)', type='filepath')
                    custom_text = gr.Textbox(
                        label='Texto a ser falado',
                        placeholder='Ex: Hello everyone! Today we are introducing our new AI-powered dubbing technology.',
                        lines=4
                    )
                    with gr.Row():
                        steps_free = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão')
                        speed_free = gr.Slider(minimum=0.5, maximum=1.5, value=1.0, step=0.1, label='Velocidade')
                    btn_free = gr.Button('Gerar Áudio com Minha Voz', variant='primary', size='lg')
                    status_free = gr.Markdown('')
                with gr.Column(scale=1):
                    audio_free_out = gr.Audio(label='Áudio Sintetizado', type='filepath', interactive=False)

            btn_free.click(
                fn=process_free_tts,
                inputs=[custom_text, ref_audio_free, steps_free, speed_free],
                outputs=[audio_free_out, status_free]
            )

demo.launch(share=True, debug=True)